In [1]:
import pandas as pd

df_sentences = pd.read_csv("train_data.csv")
df_emojis = pd.read_csv("test_data.csv")

df_emojis.head()

,sample_id,emoji_sequence
0,query_001,🕰️🕛🌑
1,query_002,👻📱💔
2,query_003,1️⃣🔵🌙
3,query_004,💀😂🤣
4,query_005,🍰🤏😎


In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("KoalaAI/Emoji-Suggester")

tokenizer = AutoTokenizer.from_pretrained("KoalaAI/Emoji-Suggester")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  14%|#3        | 115M/853M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [15]:
from tqdm import tqdm

text_emojis = []

for text in tqdm(df_sentences.text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(outputs.logits, dim=-1).item()
    emoji = model.config.id2label[pred_idx]
    text_emojis.append(emoji)

100%|████████████████████████████████████████████████████████████████████████████████| 688/688 [04:09<00:00,  2.76it/s]


In [16]:
emojis = df_emojis.emoji_sequence
emojis.head()

0     🕰️🕛🌑
1      👻📱💔
2    1️⃣🔵🌙
3      💀😂🤣
4      🍰🤏😎
Name: emoji_sequence, dtype: object

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
vect = CountVectorizer(analyzer='char')
vect.fit(list(text_emojis) + list(emojis))
X_text = vect.transform(text_emojis)
X_emojis = vect.transform(emojis)

X_text

<688x174 sparse matrix of type '<class 'numpy.int64'>'
	with 738 stored elements in Compressed Sparse Row format>

In [24]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

preds = []
for emb in X_emojis:
    sims = cosine_similarity(emb, X_text)
    best_idx = np.argmax(sims)
    sentence_id = df_sentences.iloc[best_idx]['sentence_id']
    preds.append(sentence_id)

In [29]:
df_preds = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_emojis.sample_id,
    "answer":preds
})

df_preds

,subtaskID,datapointID,answer
0,1,query_001,1010
1,1,query_002,1000
2,1,query_003,1010
3,1,query_004,1004
4,1,query_005,1000
...,...,...,...
59,1,query_060,1010
60,1,query_061,1010
61,1,query_062,1010
62,1,query_063,1000


In [30]:
df_preds.to_csv("submission.csv", index=False)